# Week 10: Agentic AI — Foundations
**Applied Generative AI**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Define what makes an AI agent** — Distinguish agents from chatbots and understand the agent taxonomy
2. **Implement tool use and function calling** — Define tools, use an LLM to select tools, and build a basic ReAct loop
3. **Understand memory and planning** — Conversation memory (buffer, summary, window), working memory, and how memory affects agent behavior
4. **Build a simple agent with LangChain/LangGraph** — Customer service agent with state, tools, conditional logic, and visualization
5. **Identify safety considerations** — Infinite loops, unintended actions, resource exhaustion; guardrails and safety wrappers

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q langchain langchain-community langchain-openai langgraph openai
# For graph visualization in Colab: !apt-get -qq install -y graphviz libgraphviz-dev && !pip install -q pygraphviz

In [ ]:
import os
from getpass import getpass

# Set OpenAI API key securely (works in Colab)
# Fallback for students without API keys: we'll use rule-based simulations where possible
if not os.environ.get("OPENAI_API_KEY"):
    try:
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key (or press Enter to skip): ")
    except Exception:
        os.environ["OPENAI_API_KEY"] = ""  # Empty = use fallbacks

HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OpenAI API key configured: {HAS_OPENAI}")

---
## 1. What Makes an AI Agent?

An **AI agent** is more than a chatbot. It can perceive, reason, and **take actions** in an environment.

### Agent = LLM + Tools + Memory + Planning

| Component | Role |
|-----------|------|
| **LLM** | Reasoning, deciding what to do next |
| **Tools** | Actions the agent can take (calculator, API calls, database queries) |
| **Memory** | Short-term (conversation) and long-term (user preferences) |
| **Planning** | Decomposing goals into steps, handling multi-step tasks |

### Agent Taxonomy (Russell & Norvig)

- **Simple reflex** — React to current percept only (if X then Y)
- **Model-based** — Maintain internal state of the world
- **Goal-based** — Plan to achieve goals
- **Utility-based** — Optimize for utility, handle trade-offs

### Agents vs. Chatbots

| Chatbot | Agent |
|---------|-------|
| Generates text only | Can execute tools (send email, query DB) |
| Stateless or limited context | Maintains memory and state |
| No external actions | Takes actions with real-world consequences |

---
## 2. Tool Use and Function Calling

Tools let an agent *do* things. The LLM decides *which* tool to call and with what arguments.

In [ ]:
# Define simple tools (no API required)
def calculator(expression: str) -> str:
    """Evaluate a math expression. Example: '2 + 3 * 4'"""
    try:
        # Safe eval for simple math only
        allowed = set("0123456789+-*/(). ")
        if not all(c in allowed for c in expression):
            return "Error: Only numbers and + - * / ( ) allowed"
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

def weather_lookup(city: str) -> str:
    """Get weather for a city (stub — returns mock data)."""
    return f"Weather in {city}: 72°F, partly cloudy (mock data)"

def web_search_stub(query: str) -> str:
    """Search the web (stub — returns mock result)."""
    return f"Search results for '{query}': [Result 1] [Result 2] (mock)"

# Tool registry: name -> (function, description)
TOOLS = {
    "calculator": (calculator, "Use for math: addition, subtraction, multiplication, division"),
    "weather": (weather_lookup, "Use for weather in a city"),
    "search": (web_search_stub, "Use for general web search"),
}

In [ ]:
# Simple rule-based tool selector (works without API)
# In production, an LLM would choose the tool from the query
def select_tool(query: str) -> tuple:
    """Select which tool to use based on query keywords."""
    q = query.lower()
    if any(w in q for w in ["weather", "temperature", "forecast", "rain"]):
        return "weather", weather_lookup
    if any(w in q for w in ["calculate", "compute", "math", "+", "-", "*", "/"]):
        return "calculator", calculator
    return "search", web_search_stub

# Test tool selection
for q in ["What's 15 * 4?", "Weather in Boston", "Who won the 2020 election?"]:
    name, fn = select_tool(q)
    print(f"Query: '{q}' -> Tool: {name}")

### ReAct Loop: Thought → Action → Observation → Thought

ReAct (Reasoning + Acting) is a standard agent loop: the model reasons, takes an action, observes the result, and repeats until done.

In [ ]:
def simple_react(query: str, max_steps: int = 3) -> str:
    """Basic ReAct loop: Thought -> Action -> Observation -> (repeat or finish)."""
    history = []
    for step in range(max_steps):
        # Thought: decide what to do
        tool_name, tool_fn = select_tool(query)
        # Action: extract args and call tool
        if tool_name == "calculator":
            # Extract simple math expression from query (allow spaces)
            import re
            exprs = re.findall(r"[\d\.\s\+\-\*\/\(\)]+", query)
            arg = max(exprs, key=len).strip() if exprs else "0"
        elif tool_name == "weather":
            # Extract city (simplified: last word or "Boston")
            words = query.split()
            arg = words[-1] if words else "New York"
        else:
            arg = query
        obs = tool_fn(arg)
        history.append((tool_name, arg, obs))
        # If we got a clear answer, we're done
        if "Error" not in obs and "mock" not in obs.lower():
            return obs
        query = obs  # Use observation as next "query" for follow-up
    return history[-1][2] if history else "No result"

In [ ]:
# Run ReAct
print(simple_react("What is 25 + 17?"))
print(simple_react("Weather in Seattle"))

---
## 3. Memory in Agents

Agents need memory to maintain context across turns. Common patterns:

- **Buffer** — Keep last N messages
- **Summary** — Compress older messages into a summary
- **Window** — Sliding window of recent messages
- **Working memory / Scratchpad** — Temporary state during a task

In [ ]:
# Simple conversation memory (buffer + window)
class ConversationMemory:
    def __init__(self, window_size: int = 5):
        self.messages = []
        self.window_size = window_size

    def add(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})

    def get_context(self) -> str:
        """Return recent messages as context string."""
        recent = self.messages[-self.window_size:]
        return "\n".join(f"{m['role']}: {m['content']}" for m in recent)

    def __len__(self):
        return len(self.messages)

mem = ConversationMemory(window_size=3)
mem.add("user", "My name is Alice")
mem.add("assistant", "Hi Alice! How can I help?")
mem.add("user", "What's the weather?")
print("Context (last 3):")

print(mem.get_context())

In [ ]:
# Working memory / scratchpad: agent can store intermediate results
class Scratchpad:
    def __init__(self):
        self.data = {}

    def write(self, key: str, value: str):
        self.data[key] = value

    def read(self, key: str) -> str:
        return self.data.get(key, "")

    def get_all(self) -> str:
        return "\n".join(f"{k}: {v}" for k, v in self.data.items())

# Example: agent stores "user_name" from earlier turn, uses it later
sp = Scratchpad()
sp.write("user_name", "Alice")
sp.write("last_query", "weather")
print("Scratchpad:", sp.get_all())

---
## 4. Building a Simple Agent — Pizza Order & Customer Service

We build a **customer service agent** using LangGraph's `StateGraph`. This expands the pizza order example with conditional edges and visualization.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# State: what flows between nodes
class PizzaState(TypedDict):
    order: str
    confirmed: bool

In [ ]:
# Node 1: Take order
def take_order(state: PizzaState):
    print("Bot: Hi! What pizza do you want?")
    choice = input("You: ")
    return {**state, "order": choice}

# Node 2: Confirm order
def confirm_order(state: PizzaState):
    print(f"Bot: You want a {state['order']} pizza, right?")
    confirmation = input("Confirm (yes/no): ")
    return {**state, "confirmed": confirmation.lower() == "yes"}

# Node 3: Finalize order
def finalize_order(state: PizzaState):
    if state["confirmed"]:
        print(f"Bot: Great! Your {state['order']} pizza is on the way!")
    else:
        print("Bot: Order was not confirmed. Let's start over.")
    return state

In [ ]:
# Build graph with conditional edges
workflow = StateGraph(PizzaState)

workflow.add_node("take_order", take_order)
workflow.add_node("confirm_order", confirm_order)
workflow.add_node("finalize_order", finalize_order)

workflow.set_entry_point("take_order")
workflow.add_edge("take_order", "confirm_order")

# Conditional: if confirmed -> finalize; else -> back to take_order
def decide_next(state: PizzaState):
    return "finalize_order" if state["confirmed"] else "take_order"

workflow.add_conditional_edges("confirm_order", decide_next, {
    "finalize_order": "finalize_order",
    "take_order": "take_order",
})
workflow.add_edge("finalize_order", END)

graph = workflow.compile()

In [ ]:
# Visualize the agent graph
from IPython.display import Image

try:
    graph_image = graph.get_graph().draw_png()
    Image(graph_image)
except Exception as e:
    print("Graphviz not installed. Run: !apt-get install graphviz && !pip install pygraphviz")
    print("Or view the graph structure:", graph.get_graph())

**Run the flow** (interactive — you'll be prompted for input):
- Try: correct pizza name + "yes" for confirmation
- Try: wrong pizza + "no" to loop back to take_order

In [ ]:
# Run the pizza order agent (interactive)
# graph.invoke({"order": "", "confirmed": False})

### Customer Service Orchestrator (Non-Interactive)

An orchestrator routes queries to the right "department" based on intent — rides, food, tickets, or fallback.

In [ ]:
import random

class ParkState(TypedDict):
    query: str
    answer: str
    selected_node: str

def orchestrator(state: ParkState):
    q = state["query"].lower()
    if "ride" in q:
        state["selected_node"] = "rides_info"
    elif "food" in q or "eat" in q:
        state["selected_node"] = "food_info"
    elif "ticket" in q or "price" in q:
        state["selected_node"] = "tickets_info"
    else:
        state["selected_node"] = "fallback"
    return state

def rides_info(state: ParkState):
    state["answer"] = f"Our park has amazing rides like {random.choice(['Roller Coaster', 'Haunted Mansion', 'Ferris Wheel'])}!"
    return state

def food_info(state: ParkState):
    state["answer"] = f"You'll love our food court! We serve {random.choice(['pizza', 'hot dogs', 'ice cream'])}."
    return state

def tickets_info(state: ParkState):
    state["answer"] = "Tickets are $20 for kids and $35 for adults."
    return state

def fallback(state: ParkState):
    state["answer"] = f"Our info desk can help with '{state['query']}'. Please visit the main desk!"
    return state

In [ ]:
# Build and run orchestrator graph
park_workflow = StateGraph(ParkState)
park_workflow.add_node("orchestrator", orchestrator)
park_workflow.add_node("rides_info", rides_info)
park_workflow.add_node("food_info", food_info)
park_workflow.add_node("tickets_info", tickets_info)
park_workflow.add_node("fallback", fallback)

park_workflow.set_entry_point("orchestrator")
park_workflow.add_conditional_edges("orchestrator", lambda s: s["selected_node"], {
    "rides_info": "rides_info", "food_info": "food_info",
    "tickets_info": "tickets_info", "fallback": "fallback",
})
for node in ["rides_info", "food_info", "tickets_info", "fallback"]:
    park_workflow.add_edge(node, END)

park_graph = park_workflow.compile()

for q in ["What rides do you have?", "Where can I eat?", "How much are the tickets?"]:
    r = park_graph.invoke({"query": q, "answer": ""})
    print(f"Q: {q}\nA: {r['answer']}\n")

---
## 5. Safety in Agentic Design

### What Can Go Wrong

- **Infinite loops** — Agent keeps calling tools without terminating
- **Unintended actions** — Wrong tool, wrong arguments, harmful effects
- **Resource exhaustion** — Too many API calls, token limits, cost overruns

### Guardrails

- **Max iterations** — Cap the number of tool calls per query
- **Human approval gates** — Require confirmation for sensitive actions
- **Action whitelisting** — Only allow specific tools in specific contexts

In [ ]:
# Simple safety wrapper: limit iterations and log all actions
def safe_agent_run(query: str, max_tool_calls: int = 3):
    action_log = []
    for step in range(max_tool_calls):
        tool_name, tool_fn = select_tool(query)
        arg = query.split()[-1] if tool_name == "weather" else query
        if tool_name == "calculator":
            import re
            exprs = re.findall(r"[\d\.\s\+\-\*\/\(\)]+", query)
            arg = max(exprs, key=len).strip() if exprs else "0"
        result = tool_fn(arg)
        action_log.append({"step": step + 1, "tool": tool_name, "arg": arg[:30], "result": result[:50]})
        if "Error" not in result and "mock" not in result.lower():
            break
        query = result
    print("Action log:", action_log)
    return action_log[-1]["result"] if action_log else "No result"

# Test: caps at max_tool_calls and logs every action
safe_agent_run("What is 2+2?", max_tool_calls=2)

---
## 6. Exercises

### Exercise 1: Add a New Tool

Add a `get_time` tool that returns the current time. Update `select_tool` so the agent uses it for queries like "What time is it?" Test that it selects the right tool for different queries.

In [ ]:
# Your code here
# from datetime import datetime
# def get_time(...): ...
# Update select_tool to handle "time", "clock", etc.

### Exercise 2: Conversation Memory

Implement conversation memory so the agent uses past context. For example, if the user says "My name is Bob" and then "What's the weather?", the agent could personalize: "Bob, the weather in [city] is..."

In [ ]:
# Your code here
# Use ConversationMemory to store user name, then inject into tool calls or responses

### Exercise 3: Safety Wrapper

Build a safety wrapper that **prevents the agent from calling any tool more than 3 times total** (across all tools). If the limit is reached, return a clear message and stop.

In [ ]:
# Your code here
# Modify safe_agent_run or create a new wrapper that tracks total tool calls

---
## 7. Responsible AI Reflection

> **An AI agent can take actions in the real world** — sending emails, querying databases, making API calls. Unlike a chatbot that merely generates text, an agent can cause real consequences.
>
> - What is the **minimum level of human oversight** you would require before deploying an agent that can take irreversible actions?
> - Is there a **class of actions** that agents should never be allowed to take autonomously?

*Reflect on these questions. Consider: financial transactions, data deletion, sending communications, modifying production systems.*

---
## 8. Summary & Next Steps

### Summary

- **Agents** = LLM + Tools + Memory + Planning
- **Tool use** — Define tools, select via LLM or rules, implement ReAct loop
- **Memory** — Buffer, window, summary, scratchpad
- **LangGraph** — StateGraph, nodes, edges, conditional edges, visualization
- **Safety** — Max iterations, action logging, human gates, whitelisting

### Next Steps (Week 11)

- Multi-agent systems and collaboration
- Advanced planning (ReWOO, Plan-and-Execute)
- Production deployment of agents